In [11]:
import pandas as pd
import re
import numpy as np
from tqdm import tqdm
import html
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')

tqdm.pandas()
print("Libraries imported successfully")

Libraries imported successfully


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [12]:
df = pd.read_csv("rawData/dataNews.csv")
print(f"Data loaded. Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

if 'content' not in df.columns:
    text_columns = [col for col in df.columns if any(k in col.lower() for k in ['content', 'text', 'article', 'body'])]
    if text_columns:
        df = df.rename(columns={text_columns[0]: 'content'})
        print(f"Renamed column '{text_columns[0]}' to 'content'")
    else:
        raise ValueError("No 'content' column found in dataset")

# Perbaikan: dua string terpisah, bukan satu string
additional_cols = ['title', 'publish_date']   # ← sebelumnya 'title,publish_date' (typo)
columns_to_keep = ['content'] + [col for col in additional_cols if col in df.columns]

df = df[columns_to_keep]
df['content'] = df['content'].fillna('').astype(str)
print(f"Kolom yang dipakai: {columns_to_keep}")
print(f"Data shape: {df.shape}")

Data loaded. Shape: (234, 3)
Columns: ['title', 'publish_date', 'content']
Kolom yang dipakai: ['content', 'title', 'publish_date']
Data shape: (234, 3)


In [13]:
initial_count = len(df)
df.drop_duplicates(subset=['content'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Removed {initial_count - len(df)} duplicate articles")
print(f"Remaining: {len(df)} articles")

print("\nSAMPLE CONTENT (first 500 chars):")
print("-" * 50)
print(df['content'].iloc[0][:500] + "...")

Removed 0 duplicate articles
Remaining: 234 articles

SAMPLE CONTENT (first 500 chars):
--------------------------------------------------
Korps Pemberantasan Tindak Pidana Korupsi (Kortas Tipikor) Polri mengungkap kasus dugaan korupsi pada Ditjen Energi Baru Terbarukan dan Konservasi Energi (EBTKE) Kementerian ESDM terkait pengadaan penerangan jalan umum tenaga surya (PJUTS). Polri menyebut kerugian dalam kasus ini Rp 19.522.256.578,74.

Dirtindak Kortastipidkor Brigjen Totok Suharyanto mengatakan ada tiga orang yang ditetapkan sebagai tersangka dalam kasus tersebut. Ketiga tersangka ialah AS selaku mantan Inspektur Jenderal Kemen...


In [14]:
def clean_news_text(text):
    """
    Data cleaning khusus berita:
    - Hapus elemen non-konten (header, byline, artefak web)
    - Hapus URL, angka, tanda baca, karakter non-teks
    - TIDAK melakukan case folding (dilakukan di cell terpisah)
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""

    # 1. Decode HTML entities
    text = html.unescape(text)

    # 2. Hapus elemen non-konten berita (header, byline, artefak web)
    non_content_patterns = [
        r'SCROLL TO CONTINUE WITH CONTENT',
        r'ADVERTISEMENT',
        r'Baca juga[^\n]*',
        r'Simak juga[^\n]*',
        r'Baca Juga[^\n]*',
        r'Lihat juga[^\n]*',
        r'Video[^\n]*',
        r'Foto[^\n]*',
        r'Editor\s*:\s*[^\n]*',       # byline editor
        r'Reporter\s*:\s*[^\n]*',     # byline reporter
        r'Penulis\s*:\s*[^\n]*',      # byline penulis
        r'Sumber\s*:\s*[^\n]*',       # byline sumber
        r'\|[^\|]*\|',                # pola header/navigasi: "A | B | C"
        r'Copyright.*',
        r'All rights reserved.*',
        r'©.*',
    ]
    for pattern in non_content_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # 3. Hapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # 4. Hapus angka (termasuk format mata uang dan persentase)
    text = re.sub(r'Rp\.?\s*[\d.,]+', '', text)   # hapus nominal rupiah
    text = re.sub(r'\d+[\d.,]*%?', '', text)       # hapus angka dan persentase

    # 5. Hapus tanda baca dan karakter non-teks
    #    Pertahankan spasi, jangan hapus huruf
    text = re.sub(r'[^\w\s]', ' ', text)

    # 6. Hapus token yang hanya berisi angka (sisa konversi)
    text = re.sub(r'\b\d+\b', '', text)

    # 7. Normalisasi whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [15]:
print("Applying cleaning...")
df['clean_content'] = df['content'].progress_apply(clean_news_text)
print("Cleaning completed")

# Case folding — cell logis terpisah sesuai rancangan
print("\nApplying case folding...")
df['casefolded'] = df['clean_content'].progress_apply(lambda x: x.lower())
print("Case folding completed")

print("\nBEFORE vs AFTER:")
print("ORIGINAL :", df['content'].iloc[0][:200])
print("CLEANED  :", df['clean_content'].iloc[0][:200])
print("CASEFOLDED:", df['casefolded'].iloc[0][:200])

Applying cleaning...


100%|██████████| 234/234 [00:00<00:00, 1686.65it/s]


Cleaning completed

Applying case folding...


100%|██████████| 234/234 [00:00<00:00, 229261.19it/s]

Case folding completed

BEFORE vs AFTER:
ORIGINAL : Korps Pemberantasan Tindak Pidana Korupsi (Kortas Tipikor) Polri mengungkap kasus dugaan korupsi pada Ditjen Energi Baru Terbarukan dan Konservasi Energi (EBTKE) Kementerian ESDM terkait pengadaan pen
CLEANED  : Korps Pemberantasan Tindak Pidana Korupsi Kortas Tipikor Polri mengungkap kasus dugaan korupsi pada Ditjen Energi Baru Terbarukan dan Konservasi Energi EBTKE Kementerian ESDM terkait pengadaan peneran
CASEFOLDED: korps pemberantasan tindak pidana korupsi kortas tipikor polri mengungkap kasus dugaan korupsi pada ditjen energi baru terbarukan dan konservasi energi ebtke kementerian esdm terkait pengadaan peneran


In [16]:
news_normalization_dict = {
    # Singkatan institusi (setelah lowercase, cocok langsung)
    'kpk'           : 'komisi pemberantasan korupsi',
    'polri'         : 'kepolisian republik indonesia',
    'esdm'          : 'energi sumber daya mineral',
    'ditjen'        : 'direktorat jenderal',
    'ebtke'         : 'energi baru terbarukan konservasi energi',
    'pjuts'         : 'penerangan jalan umum tenaga surya',
    'ppk'           : 'pejabat pembuat komitmen',
    'kpa'           : 'kuasa pengguna anggaran',
    'uu'            : 'undang undang',
    'kuhp'          : 'kitab undang undang hukum pidana',
    'pt'            : 'perseroan terbatas',
    # Istilah hukum & korupsi
    'tipikor'       : 'tindak pidana korupsi',
    'mark up'       : 'markup',
    'mark-up'       : 'markup',
    'blokir'        : 'pemblokiran',
    'post-bidding'  : 'pasca pelelangan',
    'review ulang'  : 'tinjauan ulang',
    # Singkatan umum
    'dll'           : 'dan lain lain',
    'dkk'           : 'dan kawan kawan',
    'dst'           : 'dan seterusnya',
    'yg'            : 'yang',
    'dgn'           : 'dengan',
    'krn'           : 'karena',
    'sdh'           : 'sudah',
    'utk'           : 'untuk',
    'pd'            : 'pada',
}

# Load kamus eksternal jika ada
try:
    kamus_file = pd.read_excel("kamuskatabaku.xlsx")
    file_dict = dict(zip(
        kamus_file['tidak_baku'].str.lower(),
        kamus_file['kata_baku'].str.lower()
    ))
    news_normalization_dict.update(file_dict)
    print(f"Loaded {len(file_dict)} entries from kamuskatabaku.xlsx")
except Exception as e:
    print(f"External dictionary not loaded: {e}")

print(f"Total normalization entries: {len(news_normalization_dict)}")

def normalize_news_text(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    result = []
    for word in words:
        replaced = news_normalization_dict.get(word, word)
        # Hapus kata berulang: sangat-sangat → sangat
        replaced = re.sub(r'^(\w+)-\1$', r'\1', replaced)
        if replaced.strip():        # buang hasil penggantian kosong
            result.append(replaced)
    return ' '.join(result)

print("Applying normalization...")
df['normalized'] = df['casefolded'].progress_apply(normalize_news_text)
print("Normalization completed")

Loaded 4347 entries from kamuskatabaku.xlsx
Total normalization entries: 4367
Applying normalization...


100%|██████████| 234/234 [00:00<00:00, 1567.82it/s]

Normalization completed


In [17]:
print("Tokenizing...")
df['tokens'] = df['normalized'].progress_apply(
    lambda x: word_tokenize(x) if isinstance(x, str) and len(x) > 0 else []
)
print(f"Tokenization completed")
print(f"Rata-rata token per artikel: {df['tokens'].apply(len).mean():.1f}")

Tokenizing...


100%|██████████| 234/234 [00:00<00:00, 1146.57it/s]

Tokenization completed
Rata-rata token per artikel: 404.7


In [20]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import (
    StopWordRemoverFactory, StopWordRemover, ArrayDictionary
)

sastrawi_stopwords = set(StopWordRemoverFactory().get_stop_words())

# Tambahan stopword umum yang tidak bermakna
additional_stopwords = {
    'yuk', 'nih', 'deh', 'sih', 'dong', 'kok', 'loh', 'lah',
    'btw', 'omg', 'wkwk', 'haha', 'hehe', 'hihi',
    'via', 'cc', 'rt', 'dm', 'ot',           # artefak Twitter
    'thread', 'twit', 'tweet', 'retweet',
    'like', 'share', 'follow', 'subscribe',
    'http', 'https', 'pic', 'com',            # sisa URL yang lolos
}

keep_words_news = {
    'korupsi', 'uang', 'negara', 'proyek', 'lelang', 'pengadaan',
    'tersangka', 'saksi', 'pasal', 'pidana', 'hukum', 'kerugian',
    'kontrak', 'miliar', 'rupiah', 'tidak', 'bukan', 'tanpa',
}

final_stopwords = (sastrawi_stopwords | additional_stopwords) - keep_words_news

print(f"Sastrawi stopwords  : {len(sastrawi_stopwords)}")
print(f"Setelah penambahan  : {len(sastrawi_stopwords | additional_stopwords)}")
print(f"Setelah pengurangan : {len(final_stopwords)}")

for word in keep_words_news:
    status = "AMAN" if word not in final_stopwords else "MASALAH"
    print(f"  '{word}': {status}")

def remove_stopwords(tokens):
    return [w for w in tokens if w not in final_stopwords and len(w) > 2]

print("\nRemoving stopwords...")
df['tokens_clean'] = df['tokens'].progress_apply(remove_stopwords)
print(f"Rata-rata token sebelum : {df['tokens'].apply(len).mean():.1f}")
print(f"Rata-rata token sesudah : {df['tokens_clean'].apply(len).mean():.1f}")

Sastrawi stopwords  : 809
Setelah penambahan  : 837
Setelah pengurangan : 834
  'tidak': AMAN
  'pasal': AMAN
  'tersangka': AMAN
  'korupsi': AMAN
  'hukum': AMAN
  'pengadaan': AMAN
  'kerugian': AMAN
  'kontrak': AMAN
  'lelang': AMAN
  'bukan': AMAN
  'pidana': AMAN
  'uang': AMAN
  'negara': AMAN
  'proyek': AMAN
  'miliar': AMAN
  'saksi': AMAN
  'rupiah': AMAN
  'tanpa': AMAN

Removing stopwords...


100%|██████████| 234/234 [00:00<00:00, 23376.61it/s]

Rata-rata token sebelum : 404.7
Rata-rata token sesudah : 237.6


In [21]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Kata yang tidak distem (bentuk sudah final)
no_stem_words = {
    'korupsi', 'koruptor', 'polri', 'rupiah', 'miliar', 'triliun',
    'tersangka', 'saksi', 'pidana', 'hukum', 'pasal', 'lelang',
    'pengadaan', 'kontrak', 'proyek', 'kerugian'
}

def stem_news_tokens(tokens):
    return [
        token if (token in no_stem_words or len(token) <= 3)
        else stemmer.stem(token)
        for token in tokens
    ]

print("Applying stemming...")
df['stemmed'] = df['tokens_clean'].progress_apply(stem_news_tokens)
df['stemmed_str'] = df['stemmed'].apply(lambda x: ' '.join(x))
print("Stemming completed")

Applying stemming...


100%|██████████| 234/234 [00:00<00:00, 280.03it/s]

Stemming completed


In [22]:
# Buang artikel kosong setelah preprocessing
empty = df['stemmed'].apply(lambda x: len(x) == 0).sum()
print(f"Artikel kosong setelah preprocessing: {empty}")
if empty > 0:
    df = df[df['stemmed'].apply(len) > 0].reset_index(drop=True)

print(f"\nTotal artikel: {len(df)}")
df['orig_word_count']  = df['content'].apply(lambda x: len(str(x).split()))
df['final_word_count'] = df['stemmed_str'].apply(lambda x: len(x.split()))
reduksi = (1 - df['final_word_count'].mean() / df['orig_word_count'].mean()) * 100
print(f"Rata-rata kata asli   : {df['orig_word_count'].mean():.0f}")
print(f"Rata-rata kata akhir  : {df['final_word_count'].mean():.0f}")
print(f"Reduksi               : {reduksi:.1f}%")

print("\nTOP 20 KATA:")
all_tokens = [t for tokens in df['stemmed'] for t in tokens]
for word, freq in pd.Series(all_tokens).value_counts().head(20).items():
    print(f"  {word:25s}: {freq}")

print("\nCONTOH PIPELINE (artikel pertama):")
print(f"  Original  : {df['content'].iloc[0][:120]}...")
print(f"  Cleaned   : {df['clean_content'].iloc[0][:120]}...")
print(f"  Casefolded: {df['casefolded'].iloc[0][:120]}...")
print(f"  Normalized: {df['normalized'].iloc[0][:120]}...")
print(f"  Tokens    : {df['tokens_clean'].iloc[0][:10]}")
print(f"  Stemmed   : {df['stemmed'].iloc[0][:10]}")

Artikel kosong setelah preprocessing: 0

Total artikel: 234
Rata-rata kata asli   : 417
Rata-rata kata akhir  : 238
Reduksi               : 42.9%

TOP 20 KATA:
  etanol                   : 1164
  energi                   : 784
  bbm                      : 728
  bahan                    : 702
  tidak                    : 657
  pertamina                : 619
  bakar                    : 597
  indonesia                : 427
  campur                   : 403
  sumber                   : 379
  kandung                  : 349
  daya                     : 349
  spbu                     : 346
  hasil                    : 312
  bahlil                   : 305
  tahun                    : 293
  perintah                 : 281
  bensin                   : 280
  minyak                   : 278
  mineral                  : 273

CONTOH PIPELINE (artikel pertama):
  Original  : Korps Pemberantasan Tindak Pidana Korupsi (Kortas Tipikor) Polri mengungkap kasus dugaan korupsi pada Ditjen Energi Baru...
  Cle

In [23]:
df.to_csv("cleanData/LDA_processed_news_data.csv", index=False, encoding='utf-8')
print(f"File disimpan. Total artikel: {len(df)}")

File disimpan. Total artikel: 234
